In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
PROJECT_ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_service.forecast_core import (
    PROJECT_BENCHMARK_ACCURACY,
    backtest_models,
    build_features,
    generate_product_forecast,
    get_zone_series,
    load_hourly_demand,
    resolve_raw_data_dir,
)

DATA_DIR = resolve_raw_data_dir(PROJECT_ROOT / 'model_service' / 'forecast_core.py')
TRIP_FILES = sorted(DATA_DIR.glob('*.parquet'))
ZONE_FILE = DATA_DIR / 'taxi_zone_lookup (1).csv'

TRIP_FILES, ZONE_FILE


In [ ]:
hourly_demand = load_hourly_demand(DATA_DIR)
hourly_demand.head()


In [ ]:
zone_id = 161
horizon = 'hourly'
zone_series = get_zone_series(zone_id, horizon=horizon, raw_data_dir=DATA_DIR)
features_df = build_features(zone_series, horizon=horizon)
backtest = backtest_models(features_df, horizon=horizon)

{
    'best_model': backtest['best_model'],
    'best_accuracy': backtest['best_accuracy'],
    'project_benchmark_accuracy': PROJECT_BENCHMARK_ACCURACY,
}


In [ ]:
zone_id = 161
horizon = 'hourly'
requested_date = '2025-08-01'
requested_time = '17:00'

product_forecast = generate_product_forecast(
    zone_id=zone_id,
    horizon=horizon,
    requested_date=requested_date,
    requested_time=requested_time,
    raw_data_dir=DATA_DIR,
)

summary = {
    'zone_id': zone_id,
    'horizon': horizon,
    'requested_window': product_forecast['requested_window'],
    'peak_window': product_forecast['peak_window'],
    'model_meta': product_forecast['meta'],
}

summary
